<a href="https://colab.research.google.com/github/romavallejo/TC3009C.600_AIClass/blob/main/Week1_handson_ingest_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 Hands-on: Data Ingestion with Google Cloud

## Big Data and Artificial Intelligence

**Professor:** Oscar Fuentes  
**Institution:** Tecnológico de Monterrey — Campus Ciudad de México

---

## About this notebook

In this hands-on exercise, we will build a small cloud data pipeline using real airline on-time performance data from the **U.S. Bureau of Transportation Statistics (BTS)**.

We will follow the data from its original source to an analytical database:

```text
BTS → Pandas → Parquet → Google Cloud Storage → BigQuery → SQL
```

During the exercise, you will:

1. Download a real public dataset and two supporting catalogs.
2. Inspect the data using Pandas.
3. Convert the data to the Parquet format.
4. Store the files in a Google Cloud Storage bucket.
5. Create a BigQuery dataset and three related tables.
6. Query the tables using SQL.
7. Use `LEFT JOIN` to translate airline and airport identifiers into readable names.

The objective is not only to obtain a result. It is to understand how raw files become related tables that can be explored and analyzed in the cloud.


In [1]:
# 1. Autenticación con Google Cloud
from google.colab import auth
auth.authenticate_user()

# 2. Definir tu ID de proyecto de GCP
PROJECT_ID = "bigdata-505300"  # <--- Cada alumno pone su ID de proyecto

from google.cloud import bigquery
import pandas as pd
import requests
import io
import zipfile
### No le gustan las mayúsculas al cliente
client = bigquery.Client(project=PROJECT_ID.lower())

In [2]:
# Enlace directo al catálogo de aeropuertos del BTS
airports_url = (
    "https://www.transtats.bts.gov/Download_Lookup.asp"
    "?Y11x72=Y_NVecbeg"
)

print("Descargando catálogo de aeropuertos del BTS...")

airports_response = requests.get(airports_url)
airports_response.raise_for_status()

# Leer el CSV directamente desde memoria
airports_csv = io.BytesIO(airports_response.content)
df_airports = pd.read_csv(airports_csv,
                          encoding="latin-1"
                          )

# Limpiar los nombres de las columnas
df_airports.columns = df_airports.columns.str.strip()

print(f"¡Éxito! Cargados {len(df_airports):,} aeropuertos en memoria.")
print(f"Columnas encontradas: {df_airports.columns.tolist()}")

display(df_airports.head())

Descargando catálogo de aeropuertos del BTS...
¡Éxito! Cargados 6,929 aeropuertos en memoria.
Columnas encontradas: ['Code', 'Description']


,Code,Description
0,01A,"Afognak Lake, AK: Afognak Lake Airport"
1,03A,"Granite Mountain, AK: Bear Creek Mining Strip"
2,04A,"Lik, AK: Lik Mining Camp"
3,05A,"Little Squaw, AK: Little Squaw Airport"
4,05K,"Port Alsworth, AK: Wilder Runway"


In [3]:
# Enlace directo al catálogo de aerolíneas del BTS
airlines_url = (
    "https://www.transtats.bts.gov/Download_Lookup.asp"
    "?Y11x72=Y_PNeeVRe_UVfgbel"
)

print("Descargando catálogo de aerolíneas del BTS...")

airlines_response = requests.get(airlines_url)
airlines_response.raise_for_status()

# Leer el CSV directamente desde memoria
airlines_csv = io.BytesIO(airlines_response.content)
df_airlines = pd.read_csv(airlines_csv)

# Limpiar los nombres de las columnas
df_airlines.columns = df_airlines.columns.str.strip()

print(f"¡Éxito! Cargadas {len(df_airlines):,} aerolíneas en memoria.")
print(f"Columnas encontradas: {df_airlines.columns.tolist()}")

display(df_airlines.head())

Descargando catálogo de aerolíneas del BTS...
¡Éxito! Cargadas 2,075 aerolíneas en memoria.
Columnas encontradas: ['Code', 'Description']


,Code,Description
0,02Q,Titan Airways (2006 - )
1,04Q,Tradewind Aviation (2006 - 2023)
2,05Q,"Comlux Aviation, AG (2006 - 2012)"
3,06Q,Master Top Linhas Aereas Ltd. (2007 - )
4,07Q,Flair Airlines Ltd. (2007 - )


In [4]:
# Enlace directo de descarga masiva del BTS (Sin interfaz web)
url = "https://www.transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_1.zip"

print("Descargando datos del BTS...")
response = requests.get(url)

# Unzip en memoria (sin guardar basura en el disco local)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]

    # Read the CSV content into a BytesIO object to allow multiple reads
    with z.open(csv_name) as f_zip:
        csv_data = io.BytesIO(f_zip.read())

    # Read only the header to get actual column names
    header_df = pd.read_csv(csv_data, nrows=0)
    actual_columns = header_df.columns.tolist()
    print(f"Actual columns found in CSV: {actual_columns}")

    # Reset the BytesIO object to the beginning for the full read
    csv_data.seek(0)

    # Now, read the full CSV without specifying usecols initially to avoid errors
    # and allow you to see all available columns.
    df = pd.read_csv(csv_data)

print(f"¡Éxito! Cargados {len(df):,} vuelos en memoria.")
display(df.head())
print("Please review the 'Actual columns found in CSV' above and let me know which columns you'd like to use.")

Descargando datos del BTS...
Actual columns found in CSV: ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDe

/tmp/ipykernel_609/1051704614.py:25: DtypeWarning: Columns (77,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


¡Éxito! Cargados 538,837 vuelos en memoria.


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2023,1,1,2,1,2023-01-02,9E,20363,9E,N605LR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1,1,3,2,2023-01-03,9E,20363,9E,N605LR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1,1,4,3,2023-01-04,9E,20363,9E,N331PQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1,1,5,4,2023-01-05,9E,20363,9E,N906XJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1,1,6,5,2023-01-06,9E,20363,9E,N337PQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Please review the 'Actual columns found in CSV' above and let me know which columns you'd like to use.


## Data Ingestion: Batch Processing

We have successfully ingested the flight performance data from the BTS (Bureau of Transportation Statistics) into a Pandas DataFrame. This method of data loading, where a complete dataset is retrieved at once, is characteristic of **batch processing**. Given the nature of the data (e.g., monthly releases), this approach is suitable for periodic updates, such as on a monthly basis, to keep the dataset current.

## Storing Data in Google Cloud Storage (GCS)

Following best practices for cloud data workflows, we will now store the ingested DataFrame in a Google Cloud Storage (GCS) bucket. This provides durable, scalable, and cost-effective storage. We'll store it as a Parquet file, which is an efficient columnar storage format often used in big data processing.

# Sanbdox users DO NOT GET free  Bukcets  (Es como comer con cubiertos, se recomienda, pero no es a fuerza.)

In [ ]:
# 3. Definir el nombre del bucket de GCS
# Puedes elegir un nombre único para tu bucket. Debe ser globalmente único.
GCS_BUCKET_NAME = f"{PROJECT_ID.lower()}-bts-flights-data"

# Importar la librería de GCS
from google.cloud import storage
storage_client = storage.Client(project=PROJECT_ID)

# Crear el bucket si no existe
try:
    bucket = storage_client.get_bucket(GCS_BUCKET_NAME)
    print(f"Bucket '{GCS_BUCKET_NAME}' ya existe.")
except Exception:
    print(f"Creando bucket '{GCS_BUCKET_NAME}'...")
    bucket = storage_client.create_bucket(GCS_BUCKET_NAME)
    print(f"Bucket '{GCS_BUCKET_NAME}' creado exitosamente.")

Bucket 'ia-2006-bts-flights-data' ya existe.


In [ ]:
# Guardar el DataFrame en el bucket de GCS como un archivo Parquet
# El nombre del archivo en el bucket
FILE_NAME = 'on_time_reporting_carrier_on_time_performance_2023_1.parquet'

# Convertir el DataFrame a Parquet en memoria y subirlo
# Es importante instalar pyarrow para que Pandas pueda manejar Parquet.
# !pip install pyarrow

# Asegúrate de que pyarrow esté instalado
try:
    import pyarrow
except ImportError:
    print("Instalando pyarrow para soporte de Parquet...")
    !pip install pyarrow
    import pyarrow

# Guardar el DataFrame como Parquet en un buffer de memoria
parquet_buffer = io.BytesIO()
df.to_parquet(parquet_buffer, index=False)
parquet_buffer.seek(0)

# Subir el archivo Parquet al bucket
blob = bucket.blob(FILE_NAME)
blob.upload_from_file(parquet_buffer, content_type='application/octet-stream')

print(f"DataFrame guardado exitosamente en gs://{GCS_BUCKET_NAME}/{FILE_NAME}")

DataFrame guardado exitosamente en gs://ia-2006-bts-flights-data/on_time_reporting_carrier_on_time_performance_2023_1.parquet


## Loading Data into BigQuery

Now that our data is stored efficiently in Google Cloud Storage, the next step is to load it into Google BigQuery. BigQuery is a fully managed, serverless data warehouse that enables super-fast SQL queries using the processing power of Google's infrastructure. Loading data into BigQuery will allow for powerful analytics and integration with other GCP services.

In [5]:
# 4. Definir el nombre del dataset y la tabla de BigQuery
BQ_DATASET_NAME = 'bts_flights_data'
BQ_TABLE_NAME = 'on_time_performance'

# Crear el dataset de BigQuery si no existe
try:
    dataset_id = f"{(PROJECT_ID).lower()}.{BQ_DATASET_NAME}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"  # Puedes cambiar la ubicación si lo necesitas
    dataset = client.create_dataset(dataset, timeout=30)  # Make an API request.
    print(f"Dataset '{BQ_DATASET_NAME}' creado en el proyecto '{PROJECT_ID}'.")
except Exception as e:
    if "Already Exists" in str(e):
        print(f"Dataset '{BQ_DATASET_NAME}' ya existe.")
    else:
        raise e

# Configurar la carga de BigQuery
job_config = bigquery.LoadJobConfig(
    autodetect=True # BigQuery intentará detectar el esquema automáticamente
)

# Cargar el DataFrame directamente a BigQuery
load_job = client.load_table_from_dataframe(
    df, # El DataFrame cargado previamente
    f"{dataset_id}.{BQ_TABLE_NAME}",
    job_config=job_config,
)

# Esperar a que el trabajo de carga se complete
load_job.result()

print(f"Tabla '{BQ_TABLE_NAME}' cargada exitosamente en el dataset '{BQ_DATASET_NAME}' desde el DataFrame.")

# Verificar el número de filas cargadas
table = client.get_table(f"{dataset_id}.{BQ_TABLE_NAME}") # Corrected to use the full dataset_id with project
print(f"Total de filas en la tabla '{BQ_TABLE_NAME}': {table.num_rows:,}")

Dataset 'bts_flights_data' ya existe.
Tabla 'on_time_performance' cargada exitosamente en el dataset 'bts_flights_data' desde el DataFrame.
Total de filas en la tabla 'on_time_performance': 1,077,674


In [ ]:
# REpetimos el proceso para los otros 2 catálogos. Aeropuertos y Aerolineas
# Configurar la carga de BigQuery
job_config = bigquery.LoadJobConfig(
    autodetect=True # BigQuery intentará detectar el esquema automáticamente
)

# Cargar el DataFrame de aerolíneas directamente a BigQuery
airlines_table_id = (
    f"{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines"
)

load_job = client.load_table_from_dataframe(
    df_airlines, # El DataFrame cargado previamente
    airlines_table_id,
    job_config=job_config,
)

# Esperar a que el trabajo de carga se complete
load_job.result()

airlines_table = client.get_table(airlines_table_id)

print(
    f"Tabla 'airlines' cargada exitosamente en el dataset "
    f"'{BQ_DATASET_NAME}' desde el DataFrame."
)
print(
    f"Total de filas en la tabla 'airlines': "
    f"{airlines_table.num_rows:,}"
)

Tabla 'airlines' cargada exitosamente en el dataset 'bts_flights_data' desde el DataFrame.
Total de filas en la tabla 'airlines': 2,075


# TODO  Create a table  called airports in a similar way

In [ ]:
# Configurar la carga de BigQuery
job_config = bigquery.LoadJobConfig(
    autodetect=True # BigQuery intentará detectar el esquema automáticamente
)

# Cargar el DataFrame de aeropuertos directamente a BigQuery
airports_table_id = (
    f"{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports"
)

load_job = client.load_table_from_dataframe(
    df_airports, # El DataFrame cargado previamente
    airports_table_id,
    job_config=job_config,
)

# Esperar a que el trabajo de carga se complete
load_job.result()

airports_table = client.get_table(airports_table_id)

print(
    f"Tabla 'airports' cargada exitosamente en el dataset "
    f"'{BQ_DATASET_NAME}' desde el DataFrame."
)
print(
    f"Total de filas en la tabla 'airports': "
    f"{airports_table.num_rows:,}"
)

Tabla 'airports' cargada exitosamente en el dataset 'bts_flights_data' desde el DataFrame.
Total de filas en la tabla 'airports': 6,929


The airports catalog has now been successfully loaded into BigQuery, following the same process as the airlines catalog.

## Querying Data from BigQuery

Now that the data is in BigQuery, you can query it using SQL. You can do this directly in the BigQuery Studio UI, or you can query it from a Colab notebook using the `pandas_gbq` library.

In [6]:
import pandas_gbq

# Construir la referencia completa de la tabla de BigQuery
full_table_id = f"{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.{BQ_TABLE_NAME}"

# Definir la consulta SQL
# Vamos a seleccionar algunas columnas y mostrar las primeras filas
query = f"""
SELECT
    FlightDate,
    Reporting_Airline,
    Origin,
    Dest,
    DepDelay,
    ArrDelay
FROM
    `{full_table_id}`
LIMIT 100
"""

# Ejecutar la consulta y cargar los resultados en un DataFrame de Pandas
df_from_bigquery = pandas_gbq.read_gbq(query, project_id=PROJECT_ID.lower())

# Mostrar las primeras filas del DataFrame
display(df_from_bigquery.head())

Downloading: 100%|██████████|


,FlightDate,Reporting_Airline,Origin,Dest,DepDelay,ArrDelay
0,2023-01-02,9E,ATL,ABE,-4.0,-12.0
1,2023-01-03,9E,ATL,ABE,4.0,-15.0
2,2023-01-04,9E,ATL,ABE,14.0,3.0
3,2023-01-05,9E,ATL,ABE,0.0,-5.0
4,2023-01-06,9E,ATL,ABE,0.0,-3.0


In [7]:
df_from_bigquery.describe()

,DepDelay,ArrDelay
count,100.000000,100.000000
mean,8.510000,6.420000
std,68.071634,68.735649
min,-30.000000,-31.000000
25%,-6.000000,-12.000000
50%,-3.000000,-3.500000
75%,0.000000,4.000000
max,647.000000,648.000000


## Creating a BigQuery View

A BigQuery view is a virtual table defined by a SQL query. It doesn't store data itself but executes the query every time it's referenced, providing a dynamic and up-to-date representation of your data. This is useful for pre-defining complex queries or specific subsets of data that you'll use frequently.

In [8]:
from google.cloud import bigquery

# Re-initialize the client if needed (assuming PROJECT_ID is still available)
client = bigquery.Client(project=PROJECT_ID.lower())

# Define the view name
BQ_VIEW_NAME = 'on_time_performance_sample_view'
view_id = f"{client.project}.{BQ_DATASET_NAME}.{BQ_VIEW_NAME}"

# Create a View object with the SQL query
view = bigquery.Table(view_id)
view.view_query = query.replace('LIMIT 100', '') # Remove LIMIT for the view definition

# Make an API request to create the view
try:
    view = client.create_table(view)
    print(f"Successfully created view {view.project}.{view.dataset_id}.{view.table_id}")
except Exception as e:
    if "Already Exists" in str(e):
        print(f"View '{BQ_VIEW_NAME}' already exists. Updating it.")
        view = client.update_table(view, ["view_query"])
        print(f"Successfully updated view {view.project}.{view.dataset_id}.{view.table_id}")
    else:
        raise e

print("View created successfully. You can now query it like a regular table.")

Successfully created view bigdata-505300.bts_flights_data.on_time_performance_sample_view
View created successfully. You can now query it like a regular table.


In [9]:
# Verify the view by querying it
import pandas_gbq

query_view = f"SELECT * FROM `{view_id}` LIMIT 5"
df_from_view = pandas_gbq.read_gbq(query_view, project_id=PROJECT_ID.lower())
display(df_from_view.head())

Downloading: 100%|██████████|


,FlightDate,Reporting_Airline,Origin,Dest,DepDelay,ArrDelay
0,2023-01-02,9E,ATL,ABE,-4.0,-12.0
1,2023-01-03,9E,ATL,ABE,4.0,-15.0
2,2023-01-04,9E,ATL,ABE,14.0,3.0
3,2023-01-05,9E,ATL,ABE,0.0,-5.0
4,2023-01-06,9E,ATL,ABE,0.0,-3.0


# So conceptually:

- Table: stores rows physically.
- Query: produces a temporary result.
- View: saves the query and presents its result as a reusable virtual table.

# SQL JOIN Exercises

We now have three related tables in BigQuery:

- `on_time_performance`: one row per flight
- `airlines`: airline codes and descriptive names
- `airports`: airport codes and descriptive names

The objective is to enrich the flight records without losing or accidentally duplicating observations.


## Exercise 1: Add the airline name

The flight table contains `Reporting_Airline`, but the identifier alone is difficult to interpret. Use a `LEFT JOIN` to add the descriptive airline name.

> **Important:** The BTS airline catalog is historical. The same code can appear more than once for different periods. For now, observe the result and ask: did the join preserve one result row per flight?


In [12]:
# Exercise 1: Join flights with the historical airline catalog

queryPersonal = f"""
SELECT
  p.FlightDate,
  p.Reporting_Airline,
  a.Description AS AirlineName,
  p.Origin,
  p.Dest,
  p.DepDelay,
  p.ArrDelay
FROM
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` as p
LEFT JOIN
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines` as a ON p.Reporting_Airline = a.Code
LIMIT 10
"""

query = f"""
SELECT
    f.FlightDate,
    f.Reporting_Airline,
    a.Description AS AirlineName,
    f.Origin,
    f.Dest,
    f.DepDelay,
    f.ArrDelay
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` AS f
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines` AS a
ON
    f.Reporting_Airline = a.Code
LIMIT 10
"""

df_join_airlines = pandas_gbq.read_gbq(queryPersonal, project_id=PROJECT_ID.lower())
display(df_join_airlines)


Downloading: 100%|██████████|


,FlightDate,Reporting_Airline,AirlineName,Origin,Dest,DepDelay,ArrDelay
0,2023-01-02,9E,Endeavor Air Inc. (2013 - ),ATL,ABE,-4.0,-12.0
1,2023-01-02,9E,Pinnacle Airlines Inc. (2002 - 2013),ATL,ABE,-4.0,-12.0
2,2023-01-03,9E,Endeavor Air Inc. (2013 - ),ATL,ABE,4.0,-15.0
3,2023-01-03,9E,Pinnacle Airlines Inc. (2002 - 2013),ATL,ABE,4.0,-15.0
4,2023-01-04,9E,Endeavor Air Inc. (2013 - ),ATL,ABE,14.0,3.0
5,2023-01-04,9E,Pinnacle Airlines Inc. (2002 - 2013),ATL,ABE,14.0,3.0
6,2023-01-05,9E,Endeavor Air Inc. (2013 - ),ATL,ABE,0.0,-5.0
7,2023-01-05,9E,Pinnacle Airlines Inc. (2002 - 2013),ATL,ABE,0.0,-5.0
8,2023-01-06,9E,Endeavor Air Inc. (2013 - ),ATL,ABE,0.0,-3.0
9,2023-01-06,9E,Pinnacle Airlines Inc. (2002 - 2013),ATL,ABE,0.0,-3.0


## Why did some flights appear twice?

A join does not automatically guarantee one output row per input row. If one airline code matches two rows in the catalog, the flight is returned twice.

For example, code `9E` may match both **Pinnacle Airlines** and **Endeavor Air** because the catalog preserves historical names. Before using a catalog as the right side of a join, inspect whether its key is unique.


In [13]:
# Exercise 2: Find airline codes that appear more than once

queryPersonal = f"""
SELECT
  Code,
  COUNT(*) as n
FROM
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines`
GROUP BY Code
HAVING COUNT(*) > 1
ORDER BY COUNT(*) DESC
LIMIT 10
"""

query = f"""
SELECT
    Code,
    COUNT(*) AS NumberOfNames
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines`
GROUP BY
    Code
HAVING
    COUNT(*) > 1
ORDER BY
    NumberOfNames DESC, Code
"""

df_duplicate_airline_codes = pandas_gbq.read_gbq(
    queryPersonal,
    project_id=PROJECT_ID.lower()
)

display(df_duplicate_airline_codes)


Downloading: 100%|██████████|


,Code,n
0,JX,4
1,QK,4
2,EV,4
3,7H,4
4,PA,4
5,VC,4
6,YX,4
7,BRQ,3
8,CP,3
9,CDQ,3


## An important observation about real catalogs

The BTS airline catalog is historical. A code such as `9E` can appear with more than one airline name because the company changed its name over time.

That means a `JOIN` can occasionally produce more than one result for the same flight. This is not a syntax error: it is a **data-quality and history problem**.

For now, we will:

- Keep the catalog as it was published.
- Recognize that identifiers are not always as simple as they appear.
- Use `Reporting_Airline` for the following airline comparison.

Later in the course, we can learn how to select the correct historical record using dates. We do not need to solve that problem in our first SQL exercise.


In [14]:
# Exercise 3: Inspect the names associated with one airline code

query = f"""
SELECT
    Code,
    Description
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines`
WHERE
    Code = '9E'
ORDER BY
    Description
"""

df_airline_history = pandas_gbq.read_gbq(
    query,
    project_id=PROJECT_ID.lower()
)

display(df_airline_history)


Downloading: 100%|██████████|


,Code,Description
0,9E,Endeavor Air Inc. (2013 - )
1,9E,Pinnacle Airlines Inc. (2002 - 2013)


## Understanding `LEFT JOIN`

A `JOIN` combines rows from two tables using related columns. Here, the flight table is the **left table** and the airline catalog is the **right table**.

```sql
FROM on_time_performance AS f
LEFT JOIN current_airlines AS a
ON f.Reporting_Airline = a.Code
```

The `ON` condition tells BigQuery how the rows correspond:

```text
flight.Reporting_Airline = airline.Code
```

A `LEFT JOIN` preserves every row from the left table:

- When a matching code exists, BigQuery adds the airline information.
- When no match exists, the flight remains and the added columns contain `NULL`.
- An `INNER JOIN`, in contrast, removes flights without a matching catalog entry.

### Why aliases matter

The aliases `f` and `a` identify the source of each column:

```sql
f.Reporting_Airline
a.Code
a.Description
```

### One important warning

`LEFT JOIN` guarantees that unmatched left rows are preserved, but it does **not** guarantee the same number of output rows. If one key matches several catalog rows, the left row is repeated. Always inspect the uniqueness of the key on the right side.


## Exercise 4: Flights and average delays by airline code

To avoid mixing the historical names, use the airline identifier directly from the flight table.

Calculate:

- Total flights
- Average departure delay
- Average arrival delay

Order the result from the highest to the lowest average arrival delay.

### Questions

1. Which airline code had the highest average arrival delay?
2. Does it also have the highest average departure delay?
3. Why should we consider the number of flights before comparing airlines?


In [19]:
# Exercise 4: Flights and average delays by airline code

myQuery = f"""
SELECT
  Reporting_Airline,
  COUNT(*) AS TotalFlights,
  AVG(ArrDelay) AS AverageArrivalDelay,
  AVG(DepDelay) AS AverageDepartureDelay,
FROM `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance`
GROUP BY Reporting_Airline
ORDER BY AverageArrivalDelay DESC
LIMIT 10
"""


query = f"""
SELECT
    Reporting_Airline,
    COUNT(*) AS TotalFlights,
    ROUND(AVG(DepDelay), 2) AS AverageDepartureDelay,
    ROUND(AVG(ArrDelay), 2) AS AverageArrivalDelay
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance`
GROUP BY
    Reporting_Airline
ORDER BY
    AverageArrivalDelay DESC
"""

df_airline_delays = pandas_gbq.read_gbq(
    myQuery,
    project_id=PROJECT_ID.lower()
)

display(df_airline_delays)


Downloading: 100%|██████████|


,Reporting_Airline,TotalFlights,AverageArrivalDelay,AverageDepartureDelay
0,F9,26570,21.108980,26.052259
1,G4,17230,15.802548,17.502411
2,NK,43752,13.895166,18.594974
3,OO,100694,11.114660,13.803616
4,AA,149998,10.787078,14.610332
5,B6,46498,10.552137,16.881086
6,HA,13394,9.131046,8.987779
7,UA,113314,8.315693,14.085232
8,MQ,37698,7.322892,8.646694
9,9E,33852,7.204544,12.109174


## Exercise 5: Add the origin airport name

The airport catalog uses three-letter codes in `Code`, so the correct relationship is:

```text
on_time_performance.Origin = airports.Code
```

Use a `LEFT JOIN` to translate codes such as `ATL` into readable airport descriptions.


In [21]:
# Exercise 5: Add the origin airport name

myQuery = f"""
select
  f.FlightDate,
  b.Description,
  f.Origin,
  a.Description AS OriginAirportName,
  f.Dest,
  f.DepDelay,
  f.ArrDelay
from
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` as a
left join
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` as f
on
  a.Code = f.Origin
left join
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airlines` as b
on
  f.Reporting_Airline = b.Code
limit 10
"""

query = f"""
SELECT
    f.FlightDate,
    f.Reporting_Airline,
    f.Origin,
    origin_airport.Description AS OriginAirportName,
    f.Dest,
    f.DepDelay,
    f.ArrDelay
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` AS f
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS origin_airport
ON
    f.Origin = origin_airport.Code
LIMIT 100
"""

df_origin_airports = pandas_gbq.read_gbq(myQuery, project_id=PROJECT_ID.lower())
display(df_origin_airports)


Downloading: 100%|██████████|


,FlightDate,Description,Origin,OriginAirportName,Dest,DepDelay,ArrDelay
0,2023-01-02,Endeavor Air Inc. (2013 - ),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,-4.0,-12.0
1,2023-01-02,Pinnacle Airlines Inc. (2002 - 2013),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,-4.0,-12.0
2,2023-01-03,Endeavor Air Inc. (2013 - ),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,4.0,-15.0
3,2023-01-03,Pinnacle Airlines Inc. (2002 - 2013),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,4.0,-15.0
4,2023-01-04,Endeavor Air Inc. (2013 - ),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,14.0,3.0
5,2023-01-04,Pinnacle Airlines Inc. (2002 - 2013),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,14.0,3.0
6,2023-01-05,Endeavor Air Inc. (2013 - ),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,0.0,-5.0
7,2023-01-05,Pinnacle Airlines Inc. (2002 - 2013),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,0.0,-5.0
8,2023-01-06,Endeavor Air Inc. (2013 - ),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,0.0,-3.0
9,2023-01-06,Pinnacle Airlines Inc. (2002 - 2013),ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,0.0,-3.0


## Exercise 6: Use the airport catalog twice

Each flight has both an origin and a destination. We can join the same `airports` table twice by giving it two different aliases:

- `origin_airport` describes where the flight started.
- `destination_airport` describes where the flight ended.

This does not create two copies of the catalog. The aliases simply allow the same table to play two roles in one query.


In [22]:
# Exercise 6: Add both origin and destination airport names

query = f"""
SELECT
    f.FlightDate,
    f.Reporting_Airline,
    f.Origin,
    origin_airport.Description AS OriginAirportName,
    f.Dest,
    destination_airport.Description AS DestinationAirportName,
    f.DepDelay,
    f.ArrDelay
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` AS f
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS origin_airport
ON
    f.Origin = origin_airport.Code
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS destination_airport
ON
    f.Dest = destination_airport.Code
LIMIT 100
"""

df_routes = pandas_gbq.read_gbq(query, project_id=PROJECT_ID.lower())
display(df_routes)


Downloading: 100%|██████████|


,FlightDate,Reporting_Airline,Origin,OriginAirportName,Dest,DestinationAirportName,DepDelay,ArrDelay
0,2023-01-02,9E,ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",-4.0,-12.0
1,2023-01-03,9E,ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",4.0,-15.0
2,2023-01-04,9E,ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",14.0,3.0
3,2023-01-05,9E,ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",0.0,-5.0
4,2023-01-06,9E,ATL,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern...",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",0.0,-3.0
...,...,...,...,...,...,...,...,...
95,2023-01-15,G4,BNA,"Nashville, TN: Nashville International",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",-30.0,-31.0
96,2023-01-05,G4,BNA,"Nashville, TN: Nashville International",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",-16.0,-23.0
97,2023-01-21,G4,BNA,"Nashville, TN: Nashville International",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",-24.0,-27.0
98,2023-01-02,OH,CLT,"Charlotte, NC: Charlotte Douglas International",ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",40.0,22.0


## Exercise 7: Find the busiest routes

A route is defined by the combination of origin and destination. Count the flights for every route and show the 20 busiest routes.

### Questions

1. Is `ATL → MCO` the same route as `MCO → ATL` in this query?
2. Which route had the most flights?
3. How could cancellations affect the interpretation of this result?


In [29]:
# Exercise 7: Find the 20 busiest directional routes

myQuery = f"""
SELECT
  origin_airport.Code as OriginAirportCode,
  destination_airport.Code as DestinationAirportCode,
  COUNT(*) AS TotalFlights
FROM
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` AS f
LEFT JOIN
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS origin_airport
ON
  f.Origin = origin_airport.Code
LEFT JOIN
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS destination_airport
ON
  f.Dest = destination_airport.Code
GROUP BY
  OriginAirportCode,
  DestinationAirportCode
ORDER BY
  TotalFlights DESC
LIMIT 10
"""

query = f"""
SELECT
    origin_airport.Description AS OriginAirportName,
    destination_airport.Description AS DestinationAirportName,
    COUNT(*) AS TotalFlights,
    ROUND(AVG(f.ArrDelay), 2) AS AverageArrivalDelay
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` AS f
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS origin_airport
ON
    f.Origin = origin_airport.Code
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS destination_airport
ON
    f.Dest = destination_airport.Code
GROUP BY
    OriginAirportName,
    DestinationAirportName
ORDER BY
    TotalFlights DESC
LIMIT 20
"""

df_busiest_routes = pandas_gbq.read_gbq(myQuery, project_id=PROJECT_ID.lower())
display(df_busiest_routes)


Downloading: 100%|██████████|


,OriginAirportCode,DestinationAirportCode,TotalFlights
0,OGG,HNL,2078
1,HNL,OGG,2078
2,LAX,LAS,1930
3,LAS,LAX,1924
4,BOS,DCA,1836
5,DCA,BOS,1834
6,LAX,SFO,1800
7,SFO,LAX,1772
8,LGA,ORD,1760
9,ORD,LGA,1758


## Exercise 8: Validate unmatched airport codes

A `LEFT JOIN` makes missing catalog matches visible as `NULL`. Count how many flight rows have no matching origin or destination description.

If the count is greater than zero, investigate the unmatched codes instead of silently deleting those flights.


In [32]:
# Exercise 8: Count unmatched airport codes

myQuery = f"""
select
  COUNT(*) as MissingOriDest
from
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` as f
left join
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` as origin_airport
on
  f.Origin = origin_airport.Code
left join
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` as destination_airport
on
  f.Dest = destination_airport.Code
where
  origin_airport.Code is null
  or destination_airport.Code is null
"""

query = f"""
SELECT
    COUNT(*) AS TotalFlights,
    COUNTIF(origin_airport.Code IS NULL) AS MissingOriginMatches,
    COUNTIF(destination_airport.Code IS NULL) AS MissingDestinationMatches
FROM
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` AS f
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS origin_airport
ON
    f.Origin = origin_airport.Code
LEFT JOIN
    `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` AS destination_airport
ON
    f.Dest = destination_airport.Code
"""

df_join_validation = pandas_gbq.read_gbq(myQuery, project_id=PROJECT_ID.lower())
display(df_join_validation)


Downloading: 100%|██████████|


,MissingOriDest
0,0


## Final challenge: Create your own analytical query

Create one query that combines at least **two tables** and answers a question about the January 2023 flights.

Possible questions:

- Which origin airports had the highest average departure delay?
- Which airlines operated the most flights from a chosen airport?
- Which routes had the highest percentage of arrivals delayed by at least 15 minutes?
- Which destination airports received the most cancelled flights?

Your query must include:

1. At least one `JOIN`
2. At least one aggregation such as `COUNT`, `AVG`, or `COUNTIF`
3. A `GROUP BY`
4. An `ORDER BY`
5. A short interpretation of the result


In [34]:
# Question: - Which origin airports had the highest average departure delay on Jan 2023?

myQuery = f"""
select
  origin_airport.Description as OriginAirportName,
  AVG(f.DepDelay) as AverageDepartureDelay
from
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.on_time_performance` as f
left join
  `{PROJECT_ID.lower()}.{BQ_DATASET_NAME}.airports` as origin_airport
on
  f.Origin = origin_airport.Code
where
  f.FlightDate between '2023-01-01' and '2023-01-31'
group by
  OriginAirportName
order by
  AverageDepartureDelay desc
limit 10
"""

df_join_validation = pandas_gbq.read_gbq(myQuery, project_id=PROJECT_ID.lower())
display(df_join_validation)


Downloading: 100%|██████████|


,OriginAirportName,AverageDepartureDelay
0,"Riverton/Lander, WY: Central Wyoming Regional",66.033333
1,"Pellston, MI: Pellston Regional Emmet County",60.693878
2,"North Bend/Coos Bay, OR: Southwest Oregon Regi...",51.294118
3,"Clarksburg/Fairmont, WV: North Central West Vi...",44.200000
4,"Prescott, AZ: Prescott Regional Ernest A Love ...",44.166667
5,"Jackson, WY: Jackson Hole",44.151697
6,"Bishop, CA: Bishop Airport",39.833333
7,"Williston, ND: Williston Basin International",39.488889
8,"North Platte, NE: North Platte Regional Airpor...",39.188406
9,"Escanaba, MI: Delta County",39.120690
